## 1. Preparación

Conecto Google Drive y cargo el dataset unificado junto con el
vectorizador ya entrenado — mismo setup que en similitud de contenidos,
porque la detección de duplicados reutiliza exactamente la misma lógica
de comparación.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import joblib
import sys
import importlib

sys.path.append('/content/drive/MyDrive/Datasets_TechMind/similitud contenidos')
importlib.invalidate_caches()
from limpieza_texto import limpiar_texto

import nltk
nltk.download('stopwords')

RUTA_DATASET = '/content/drive/MyDrive/Datasets_TechMind/1 Semana - con el Dataset final con todo incluido - Repo GitHub/procesados/dataset_FINAL_UNIFICADO_techmind.csv'
RUTA_VECTORIZADOR = '/content/drive/MyDrive/Datasets_TechMind/modelo baseline/vectorizer.pkl'

df = pd.read_csv(RUTA_DATASET)
vectorizador = joblib.load(RUTA_VECTORIZADOR)

df["texto_limpio"] = df["texto"].apply(limpiar_texto)
X_todos = vectorizador.transform(df["texto_limpio"])

print("Filas del dataset:", len(df))
print("Forma de la matriz completa:", X_todos.shape)

Mounted at /content/drive


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Filas del dataset: 1400
Forma de la matriz completa: (1400, 3000)


## 2. Revisar si un texto nuevo es duplicado

Comparo un texto nuevo contra los 1.400 documentos ya guardados. Si el
más parecido supera el umbral (80% por defecto, según lo acordado en el
backlog del proyecto), se marca como posible duplicado y se devuelve el
título y el porcentaje de similitud del documento existente. El umbral
queda como parámetro, no fijo en el código, para que se pueda ajustar
sin tocar la función.

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

def chequear_duplicado(texto_nuevo, umbral=0.80):
    texto_nuevo_limpio = limpiar_texto(texto_nuevo)
    vector_nuevo = vectorizador.transform([texto_nuevo_limpio])

    similitudes = cosine_similarity(vector_nuevo, X_todos)[0]

    indice_mas_similar = similitudes.argmax()
    similitud_maxima = similitudes[indice_mas_similar]

    if similitud_maxima >= umbral:
        return {
            "es_duplicado": True,
            "similitud": round(float(similitud_maxima), 3),
            "titulo_existente": df.iloc[indice_mas_similar]["titulo"]
        }
    else:
        return {
            "es_duplicado": False,
            "similitud": round(float(similitud_maxima), 3),
            "titulo_existente": None
        }

## 3. Probar con los tres casos que pide el backlog

Pruebo con un texto idéntico a uno ya guardado, uno parecido pero no
igual, y uno claramente distinto — para confirmar que la función
responde como corresponde en los tres casos.

In [4]:
texto_identico = df.iloc[0]["texto"]  # copio el texto exacto de la primera fila del dataset

resultado_1 = chequear_duplicado(texto_identico)
print("Caso 1 (idéntico):", resultado_1)

Caso 1 (idéntico): {'es_duplicado': True, 'similitud': 1.0, 'titulo_existente': 'Concepto backend de la plataforma eSano eHealth para intervenciones basadas en Internet y dispositivos móviles'}


## Un hallazgo importante sobre cómo funciona esto

Al principio probé con un texto parafraseado (mismo contenido, escrito
con otras palabras) y el sistema NO lo detectó como duplicado — dio solo
0.415 de similitud. Después probé con el texto original más una sola
oración agregada al final, y ahí sí lo detectó bien — 0.994 de similitud.

¿Por qué pasa esto? Este sistema compara palabras exactas que se repiten
entre dos textos, no el significado de lo que dicen. Si alguien reescribe
un documento completo con otras palabras, el sistema no se da cuenta de
que es lo mismo — pero si alguien sube el mismo archivo dos veces, o con
ediciones menores (un párrafo agregado, un typo corregido), sí lo va a
detectar bien.

Para el caso de uso real del proyecto (evitar que se suba el mismo
documento dos veces, o versiones muy parecidas de un archivo), esto
funciona correctamente. La limitación aparece solo si alguien reescribe
un documento entero desde cero — un caso mucho menos común, y que
quedaría fuera del alcance de esta primera versión.

In [5]:
texto_parecido = """
Este contenido explica el concepto backend de una plataforma de eHealth
llamada eSano, pensada para intervenciones de salud basadas en Internet
y aplicaciones móviles.
"""

resultado_2 = chequear_duplicado(texto_parecido)
print("Caso 2 (parecido):", resultado_2)

Caso 2 (parecido): {'es_duplicado': False, 'similitud': 0.415, 'titulo_existente': None}


In [6]:
texto_base = df.iloc[0]["texto"]
texto_con_edicion_menor = texto_base + " Este documento fue revisado y actualizado en agosto de 2026."

resultado_2 = chequear_duplicado(texto_con_edicion_menor)
print("Caso 2 (edición menor):", resultado_2)

Caso 2 (edición menor): {'es_duplicado': True, 'similitud': 0.994, 'titulo_existente': 'Concepto backend de la plataforma eSano eHealth para intervenciones basadas en Internet y dispositivos móviles'}


## Caso 3: texto claramente distinto

Pruebo con un texto de un tema totalmente diferente al del dataset, para
confirmar que el sistema NO lo marca como duplicado cuando no debería.

In [7]:
texto_distinto = """
La fotosíntesis es el proceso mediante el cual las plantas convierten la
luz solar en energía química, utilizando dióxido de carbono y agua para
producir glucosa y oxígeno.
"""

resultado_3 = chequear_duplicado(texto_distinto)
print("Caso 3 (distinto):", resultado_3)

Caso 3 (distinto): {'es_duplicado': False, 'similitud': 0.264, 'titulo_existente': None}


## Resumen

La función chequear_duplicado() reutiliza el mismo vectorizador y la
misma lógica de similitud coseno del resto del proyecto — sin entrenar
nada nuevo. El umbral (80% por defecto) queda como parámetro, no fijo en
el código, tal como pide el backlog.

Probada con los 3 casos: texto idéntico (similitud 1.0, detectado),
texto con edición menor (similitud 0.994, detectado), y texto de otro
tema (similitud 0.264, no detectado). Limitación conocida: no detecta
reformulaciones completas de un mismo texto (documentado arriba).

Para Backend: el chequeo debe hacerse DESPUÉS de que el archivo ya esté
en OCI Object Storage (así se acordó en el equipo) — si el resultado es
es_duplicado: True, Backend necesita borrar el archivo recién subido.